In [ ]:
# Install libraries for both questions
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
!pip -q install pytesseract opencv-python-headless pillow matplotlib transformers torch

In [ ]:
# Q1. Design a Cognitive Assistant for Reading Labels of Medicine Bottles (dataset or uploaded images).

from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import pytesseract
import re

uploaded = files.upload()
image_paths = list(uploaded)

if not image_paths:
    raise ValueError("Upload at least one medicine bottle label image.")

for path in image_paths:
    image = cv2.imread(path)
    if image is None:
        print(f"Could not open {path}")
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    processed = cv2.bilateralFilter(gray, 7, 75, 75)
    processed = cv2.adaptiveThreshold(processed, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 31, 11)

    original_text = pytesseract.image_to_string(gray, config="--psm 6").strip()
    processed_text = pytesseract.image_to_string(processed, config="--psm 6").strip()
    text = max([original_text, processed_text], key=len)

    print(f"\nIMAGE: {path}")
    print("RECOGNIZED LABEL TEXT:\n", text or "No readable text found")

    for field, pattern in {
        "Possible strength": r"\b\d+(?:\.\d+)?\s?(?:mg|mcg|g|ml|%)\b",
        "Possible expiry date": r"(?:EXP(?:IRY)?|USE BEFORE|BEST BEFORE)\s*[:.\-]?\s*([A-Z0-9/\-. ]{4,18})",
        "Possible batch number": r"(?:BATCH|LOT|B\.?\s?NO\.?)\s*[:#\-]?\s*([A-Z0-9\-/]+)"
    }.items():
        found = re.findall(pattern, text, flags=re.IGNORECASE)
        print(f"{field}: {found if found else 'Not detected'}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Uploaded medicine label")
    axes[1].imshow(processed, cmap="gray")
    axes[1].set_title("Preprocessed for OCR")
    for ax in axes:
        ax.axis("off")
    plt.show()

print("\nOCR can make mistakes. Verify all medicine details against the actual package or a pharmacist.")

In [ ]:
# Q2. Design an Emotion-aware Cognitive Assistant using Facial Expression Detection.

from google.colab import files
from PIL import Image
from transformers import pipeline
import matplotlib.pyplot as plt
import numpy as np
import cv2

uploaded = files.upload()
image_paths = list(uploaded)

if not image_paths:
    raise ValueError("Upload at least one face image.")

classifier = pipeline("image-classification", model="trpakov/vit-face-expression")
face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

responses = {
    "happy": "The facial expression looks happy.",
    "sad": "The facial expression looks sad.",
    "angry": "The facial expression looks angry.",
    "surprise": "The facial expression looks surprised.",
    "surprised": "The facial expression looks surprised.",
    "fear": "The facial expression looks fearful.",
    "disgust": "The facial expression looks disgusted.",
    "neutral": "The facial expression looks neutral."
}

for path in image_paths:
    image = cv2.imread(path)
    if image is None:
        print(f"Could not open {path}")
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    print(f"\nIMAGE: {path} | Faces detected: {len(faces)}")

    annotated = cv2.cvtColor(image.copy(), cv2.COLOR_BGR2RGB)
    if len(faces) == 0:
        print("No frontal face detected. Try a clear, front-facing photo.")
    else:
        for i, (x, y, w, h) in enumerate(faces, start=1):
            face_image = Image.fromarray(cv2.cvtColor(image[y:y+h, x:x+w], cv2.COLOR_BGR2RGB))
            predictions = classifier(face_image, top_k=3)
            label = predictions[0]["label"].lower()
            score = predictions[0]["score"]
            print(f"Face {i}: {label.title()} ({score:.1%} model score)")
            print("Assistant:", responses.get(label, f"Detected expression: {label}"))
            print("Top predictions:", [(p["label"], round(p["score"], 3)) for p in predictions])
            cv2.rectangle(annotated, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(annotated, label, (x, max(20, y-8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)

    plt.figure(figsize=(8, 6))
    plt.imshow(annotated)
    plt.axis("off")
    plt.show()

print("\nFacial-expression classification is an estimate of visible expression, not a reliable measure of a person's feelings.")